# TF-IDF + Logistic Regression — Elden Ring Steam Reviews

So sánh Logistic Regression mặc định và `class_weight='balanced'`. Chỉ đánh giá và chọn cấu hình trên Validation; không tính metric trên Test.

> **Lưu ý rò rỉ dữ liệu:** các CSV chia ở bước trước có nội dung review trùng giữa các tập. Notebook này phát hiện, loại nhóm review có nhãn bất nhất, phân hoạch lại theo nội dung duy nhất (không phân biệt hoa/thường), rồi lưu các tập an toàn thay thế vào `data/processed/` trước khi huấn luyện.

In [1]:
from pathlib import Path
from time import perf_counter

import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    precision_recall_fscore_support, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split

DATA_DIR = Path('data/processed')
if not DATA_DIR.exists():
    DATA_DIR = Path('../data/processed')
SPLIT_NAMES = ['train', 'validation', 'test']
if not all((DATA_DIR / f'{name}.csv').exists() for name in SPLIT_NAMES):
    raise FileNotFoundError('Cần có train.csv, validation.csv và test.csv trong data/processed/')

source_splits = {name: pd.read_csv(DATA_DIR / f'{name}.csv') for name in SPLIT_NAMES}
for name, frame in source_splits.items():
    if not {'review', 'label'}.issubset(frame.columns):
        raise ValueError(f'{name}.csv cần có cột review và label')
    frame['label'] = frame['label'].astype(int)
print({name: len(frame) for name, frame in source_splits.items()})

{'train': 6834, 'validation': 1464, 'test': 1465}


## 1. Kiểm tra và xử lý trùng lặp giữa các tập
Kiểm tra cả nội dung chính xác và nội dung không phân biệt hoa/thường. Điều này quan trọng vì TF-IDF mặc định chuyển chữ thường. Khi có trùng lặp, đánh giá trên một tập chứa cùng review đã thấy ở tập khác sẽ quá lạc quan.

In [2]:
def review_key(text):
    # TF-IDF mặc định lowercase=True nên gom cả review chỉ khác hoa/thường.
    return str(text).casefold().strip()

overlap_rows = []
for left, right in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    left_frame = source_splits[left]
    right_frame = source_splits[right]
    exact_left = set(left_frame['review'].astype(str))
    exact_right = set(right_frame['review'].astype(str))
    key_left = set(left_frame['review'].map(review_key))
    key_right = set(right_frame['review'].map(review_key))
    overlap_rows.append({
        'Cặp tập': f'{left} – {right}',
        'Review trùng chính xác': len(exact_left & exact_right),
        'Review trùng không phân biệt hoa/thường': len(key_left & key_right),
    })
overlap_table = pd.DataFrame(overlap_rows).set_index('Cặp tập')
display(overlap_table)

# Gộp các hàng để xây dựng lại các phân hoạch theo nội dung review.
combined = pd.concat(
    [frame[['review', 'label']].assign(source_split=name)
     for name, frame in source_splits.items()],
    ignore_index=True,
)
combined['review_key'] = combined['review'].map(review_key)
group_stats = combined.groupby('review_key').agg(
    row_count=('label', 'size'), label_count=('label', 'nunique')
)
conflicting_keys = set(group_stats.index[group_stats['label_count'] > 1])
conflicting_rows = int(combined['review_key'].isin(conflicting_keys).sum())
print(f'Tổng hàng ban đầu: {len(combined):,}')
print(f'Nội dung review duy nhất (không phân biệt hoa/thường): {len(group_stats):,}')
print(f'Nhóm review có nhãn mâu thuẫn: {len(conflicting_keys)} nhóm ({conflicting_rows} hàng) — loại khỏi phân hoạch mới.')

,Review trùng chính xác,Review trùng không phân biệt hoa/thường
Cặp tập,,
train – validation,152,151
train – test,134,147
validation – test,74,74


Tổng hàng ban đầu: 9,763
Nội dung review duy nhất (không phân biệt hoa/thường): 7,964
Nhóm review có nhãn mâu thuẫn: 18 nhóm (234 hàng) — loại khỏi phân hoạch mới.


**Nhận xét:** Có review trùng giữa các phân hoạch cũ, nên không dùng trực tiếp để đánh giá. Các review trùng nhưng nhãn khác nhau cũng không có nhãn chuẩn rõ ràng; loại toàn bộ nhóm mâu thuẫn. Sau đó giữ một bản ghi cho mỗi nội dung, chia lại có phân tầng và thay thế ba CSV cũ bằng các phân hoạch không giao nhau. Đây là chia lại dữ liệu đã gán nhãn; tập Test mới vẫn chưa được dùng để tính điểm hoặc chọn mô hình.

In [3]:
clean_data = combined.loc[~combined['review_key'].isin(conflicting_keys)].drop_duplicates(
    subset='review_key', keep='first'
).copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    clean_data['review'], clean_data['label'],
    test_size=0.30, random_state=42, stratify=clean_data['label'],
)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp,
)

safe_splits = {
    'train': pd.DataFrame({'review': X_train, 'label': y_train}),
    'validation': pd.DataFrame({'review': X_validation, 'label': y_validation}),
    'test': pd.DataFrame({'review': X_test, 'label': y_test}),
}
safe_keys = {name: set(frame['review'].map(review_key))
             for name, frame in safe_splits.items()}
assert not (safe_keys['train'] & safe_keys['validation'])
assert not (safe_keys['train'] & safe_keys['test'])
assert not (safe_keys['validation'] & safe_keys['test'])

for name, frame in safe_splits.items():
    frame.reset_index(drop=True).to_csv(DATA_DIR / f'{name}.csv', index=False, encoding='utf-8')
print('Đã lưu lại ba phân hoạch không trùng review.')
print('Số mẫu:', {name: len(frame) for name, frame in safe_splits.items()})

Đã lưu lại ba phân hoạch không trùng review.
Số mẫu: {'train': 5562, 'validation': 1192, 'test': 1192}


## 2. TF-IDF
Vectorizer chỉ được `fit` trên review của Train. Validation được biến đổi bằng cùng vectorizer; không dùng Validation/Test để học từ vựng hoặc trọng số IDF.

In [4]:
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(safe_splits['train']['review'])
X_validation_tfidf = vectorizer.transform(safe_splits['validation']['review'])
y_train = safe_splits['train']['label']
y_validation = safe_splits['validation']['label']
print(f'Train TF-IDF: {X_train_tfidf.shape[0]:,} mẫu × {X_train_tfidf.shape[1]:,} đặc trưng')
print(f'Validation TF-IDF: {X_validation_tfidf.shape[0]:,} mẫu')
print('Vectorizer chỉ fit trên Train.')

Train TF-IDF: 5,562 mẫu × 11,662 đặc trưng
Validation TF-IDF: 1,192 mẫu
Vectorizer chỉ fit trên Train.


## 3. Huấn luyện và đánh giá trên Validation
Hai cấu hình dùng cùng Logistic Regression và `max_iter=1000`; cấu hình thứ hai đặt `class_weight='balanced'` để tính đến mất cân bằng lớp. Thời gian được đo riêng cho lời gọi `fit` (không gồm TF-IDF). Ma trận nhầm lẫn trình bày theo thứ tự nhãn 0, 1.

In [5]:
model_configs = {
    'Mặc định': LogisticRegression(max_iter=1000),
    'class_weight=balanced': LogisticRegression(max_iter=1000, class_weight='balanced'),
}

results = []
confusion_matrices = {}
trained_models = {}
for name, model in model_configs.items():
    start_time = perf_counter()
    model.fit(X_train_tfidf, y_train)
    training_seconds = perf_counter() - start_time
    predictions = model.predict(X_validation_tfidf)

    class_precision, class_recall, class_f1, _ = precision_recall_fscore_support(
        y_validation, predictions, labels=[0, 1], zero_division=0
    )
    results.append({
        'Cấu hình': name,
        'Accuracy': accuracy_score(y_validation, predictions),
        'Precision (lớp 1)': precision_score(y_validation, predictions, pos_label=1, zero_division=0),
        'Recall (lớp 1)': recall_score(y_validation, predictions, pos_label=1, zero_division=0),
        'F1 (lớp 1)': f1_score(y_validation, predictions, pos_label=1, zero_division=0),
        'Macro-F1': f1_score(y_validation, predictions, average='macro', zero_division=0),
        'Precision (lớp 0)': class_precision[0],
        'Recall (lớp 0)': class_recall[0],
        'F1 (lớp 0)': class_f1[0],
        'Thời gian train (giây)': training_seconds,
    })
    confusion_matrices[name] = confusion_matrix(y_validation, predictions, labels=[0, 1])
    trained_models[name] = model

results_table = pd.DataFrame(results).set_index('Cấu hình')
display(results_table.round(4))

for name, matrix in confusion_matrices.items():
    print(f'Ma trận nhầm lẫn — {name} (hàng = nhãn thật, cột = dự đoán):')
    display(pd.DataFrame(matrix, index=['Thật 0', 'Thật 1'],
                         columns=['Dự đoán 0', 'Dự đoán 1']))

,Accuracy,Precision (lớp 1),Recall (lớp 1),F1 (lớp 1),Macro-F1,Precision (lớp 0),Recall (lớp 0),F1 (lớp 0),Thời gian train (giây)
Cấu hình,,,,,,,,,
Mặc định,0.9304,0.9310,0.9991,0.9639,0.5049,0.6667,0.0238,0.0460,0.0653
class_weight=balanced,0.8842,0.9691,0.9043,0.9356,0.6827,0.3291,0.6190,0.4298,0.0660


Ma trận nhầm lẫn — Mặc định (hàng = nhãn thật, cột = dự đoán):


,Dự đoán 0,Dự đoán 1
Thật 0,2,82
Thật 1,1,1107


Ma trận nhầm lẫn — class_weight=balanced (hàng = nhãn thật, cột = dự đoán):


,Dự đoán 0,Dự đoán 1
Thật 0,52,32
Thật 1,106,1002


## 4. So sánh cấu hình và lớp 0
Accuracy và F1 lớp 1 không đủ để đánh giá dữ liệu mất cân bằng. Macro-F1 cho hai lớp trọng số bằng nhau; F1 lớp 0 cho biết khả năng phát hiện đánh giá không đề xuất. Chọn cấu hình có Macro-F1 Validation cao hơn.

In [6]:
best_name = results_table['Macro-F1'].idxmax()
best_row = results_table.loc[best_name]
print(f'Cấu hình được chọn theo Macro-F1 Validation: {best_name}')
print(f"Macro-F1 = {best_row['Macro-F1']:.4f}; F1 lớp 0 = {best_row['F1 (lớp 0)']:.4f}")
print('Không sử dụng tập Test để chọn mô hình hoặc tính metric trong notebook này.')

Cấu hình được chọn theo Macro-F1 Validation: class_weight=balanced
Macro-F1 = 0.6827; F1 lớp 0 = 0.4298
Không sử dụng tập Test để chọn mô hình hoặc tính metric trong notebook này.
